# Speech-to-Reasoning Pipeline with Whisper and a Quantized LLM

---

## Overview

This notebook implements a complete pipeline that:

1. Takes an audio clip (speech) as input.
2. Transcribes the speech to text using **OpenAI Whisper**.
3. Cleans the transcription (removes filler words, fixes spacing/punctuation).
4. Sends the cleaned transcription to a **4-bit quantized reasoning LLM** loaded with **Unsloth**.
5. Displays the model's step-by-step reasoning answer.

The notebook is fully self-contained: it generates its own sample audio clips using text-to-speech
so it can be run from top to bottom in Google Colab with no manual file uploads required. An
optional cell is also provided for uploading your own audio file.

## How to run

1. In Colab, go to **Runtime > Change runtime type** and select **T4 GPU**.
2. Run every cell from top to bottom, in order.
3. No API keys or manual configuration are required.

No emojis are used anywhere in this notebook or its outputs, per submission guidelines.


## Step 1: Install Dependencies

We install everything in one place so the environment is reproducible. Note that `unsloth_zoo`
is installed as its own explicit step — installing it only as a transitive dependency of
`unsloth` can silently fail on Colab and cause runtime import errors later in the notebook.

Packages installed:

- `openai-whisper` — speech-to-text (ASR) engine.
- `unsloth` and `unsloth_zoo` — fast, memory-efficient loading and inference of 4-bit quantized LLMs.
- `transformers`, `accelerate`, `bitsandbytes` — core Hugging Face stack used under the hood by Unsloth.
- `gTTS` — used only to generate sample speech audio for the self-contained test cases below.
- `ffmpeg` (system package) — required by Whisper to decode audio files.


In [ ]:
# Install system-level dependency required by Whisper for audio decoding
!apt-get -qq install -y ffmpeg > /dev/null

# Core speech recognition library
!pip install -q -U openai-whisper

# Unsloth for fast 4-bit quantized LLM loading and inference.
# This single command installs unsloth and its compatible dependencies,
# including a working version of transformers, accelerate, bitsandbytes, and unsloth_zoo.
!pip install "unsloth[colab-new] @ git+https://github.com/unsloth/unsloth.git"

# Text-to-speech, used only to auto-generate demo audio clips for testing
!pip install -q -U gTTS

print("All dependencies installed successfully.")

## Step 2: How Whisper Performs Automatic Speech Recognition

Before writing code, it is worth understanding what Whisper is doing internally:

- **Architecture:** Whisper is an encoder-decoder Transformer, trained on approximately 680,000
  hours of multilingual and multitask supervised audio data collected from the web.
- **Input representation:** Raw audio is resampled to 16 kHz, split into 30-second windows, and
  converted into an 80-channel log-Mel spectrogram — a visual, frequency-based representation of
  the audio signal that Transformers can process the same way they process image patches or text
  tokens.
- **Encoder:** The audio encoder consumes the spectrogram and produces a sequence of hidden
  representations that capture acoustic and phonetic information.
- **Decoder:** The text decoder is an autoregressive Transformer language model. It attends to the
  encoder's audio representations and generates output tokens (text) one at a time, conditioned on
  special task tokens that tell the model whether to transcribe, translate, detect language, etc.
- **Long audio handling:** Because the encoder only processes fixed 30-second windows, Whisper
  internally slides a window across longer audio, transcribes each window, and uses timestamp
  tokens to stitch the results back together into one continuous transcript. This is why the same
  `model.transcribe()` call can be used regardless of whether the clip is 5 seconds or 5 minutes
  long.
- **Robustness:** Because of the scale and diversity of its training data, Whisper generalizes well
  across accents, background noise, and technical vocabulary without any fine-tuning.

In this notebook we use the `base` Whisper checkpoint by default, which gives a good balance of
speed and accuracy on a free-tier T4 GPU. Larger checkpoints (`small`, `medium`, `large-v3`) can be
substituted by changing a single variable, at the cost of more GPU memory and slower inference.


## Step 3: Imports and GPU Check

We import all libraries up front and confirm that a GPU runtime is active. If no GPU is detected,
go to **Runtime > Change runtime type** and select **T4 GPU** before continuing.


In [ ]:
import os
import re
import gc
import time
import textwrap

import torch
import whisper
from gtts import gTTS
from IPython.display import Audio, display, Markdown

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU device:", torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"Total GPU memory: {total_mem:.2f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type and select a GPU (T4).")


## Step 4: Project Folder Setup

We create a small working folder structure inside the Colab runtime to keep sample audio files and
any generated outputs organized. This mirrors the folder structure described in the submission
README.


In [ ]:
BASE_DIR = "Speech_to_Reasoning"
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

os.makedirs(AUDIO_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Working directories ready:")
print(" -", AUDIO_DIR)
print(" -", OUTPUT_DIR)


## Step 5: Generate Sample Audio Clips (Self-Contained Testing)

To make this notebook runnable end-to-end without requiring any file uploads, we generate three
sample speech clips using Google Text-to-Speech (`gTTS`). Each clip contains a short question or
statement that requires some reasoning to answer, which lets us properly test the second half of
the pipeline (the reasoning LLM), not just the transcription step.

If you would rather test with your own voice recording, skip this cell and use the **optional
upload cell** later in the notebook instead.


In [ ]:
sample_texts = [
    "If a train travels sixty miles per hour for two hours, how far does it travel?",
    "Which is heavier, a kilogram of feathers or a kilogram of steel? Explain your reasoning.",
    "A farmer has seventeen sheep. All but nine die. How many sheep are left?",
]

audio_paths = []

for i, text in enumerate(sample_texts, start=1):
    path = os.path.join(AUDIO_DIR, f"sample_{i}.mp3")
    tts = gTTS(text=text, lang="en")
    tts.save(path)
    audio_paths.append(path)
    print(f"Generated {path} -> \"{text}\"")

print()
print(f"Total sample audio clips generated: {len(audio_paths)}")


### Optional: Upload Your Own Audio File

Run the cell below only if you want to test with your own recorded audio (wav, mp3, m4a, etc). It
will add your file to the same `audio_paths` list used for testing further down. If you
skip this cell, the notebook will simply continue using the three generated samples above.


In [ ]:
# Uncomment and run this cell to upload your own audio file in Google Colab.

# from google.colab import files
# uploaded = files.upload()
# for filename in uploaded.keys():
#     custom_path = os.path.join(AUDIO_DIR, filename)
#     os.rename(filename, custom_path)
#     audio_paths.append(custom_path)
#     print(f"Added custom audio file: {custom_path}")


## Step 6: Load the Whisper Model

We load the `base` Whisper checkpoint onto the GPU. This checkpoint is small enough to run
comfortably on a free-tier T4 while still giving good transcription quality for clear speech.

You can switch to a larger checkpoint (`small`, `medium`) by changing `WHISPER_MODEL_SIZE` below if
you need higher accuracy and have GPU memory to spare.


In [ ]:
WHISPER_MODEL_SIZE = "base"  # options: tiny, base, small, medium, large-v3

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Whisper '{WHISPER_MODEL_SIZE}' model on {device} ...")
whisper_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)
print("Whisper model loaded.")


## Step 7: Transcription Function

The function below wraps `whisper_model.transcribe()`. Whisper's `transcribe()` method already
handles audio of any length internally by processing it in 30-second windows and stitching the
results together, so the same function works for a 3-second clip or a 3-minute recording.

We also time the transcription so we can report inference speed.


In [ ]:
def transcribe_audio(audio_path, model=None, verbose=False):
    """
    Transcribe a single audio file using Whisper.

    Args:
        audio_path (str): Path to the audio file (wav, mp3, m4a, etc).
        model: A loaded Whisper model instance. Defaults to the global whisper_model.
        verbose (bool): If True, prints Whisper's internal progress.

    Returns:
        dict with keys: "text", "language", "duration_seconds"
    """
    if model is None:
        model = whisper_model

    start_time = time.time()
    result = model.transcribe(audio_path, verbose=verbose)
    elapsed = time.time() - start_time

    return {
        "text": result["text"].strip(),
        "language": result.get("language", "unknown"),
        "duration_seconds": round(elapsed, 2),
    }


# Quick smoke test on the first sample clip
test_result = transcribe_audio(audio_paths[0])
print("Detected language:", test_result["language"])
print("Transcription time:", test_result["duration_seconds"], "seconds")
print("Transcribed text:", test_result["text"])


## Step 8: Transcription Cleaning

Raw Whisper output is usually quite clean, but real-world audio (especially longer or noisier
recordings) can produce repeated filler words, inconsistent spacing, or stray punctuation. The
function below performs light, safe cleanup before the text is sent to the reasoning model:

- Collapses repeated whitespace.
- Removes common filler words/disfluencies (`um`, `uh`, `you know`, etc).
- Fixes spacing around punctuation.
- Ensures the text ends with proper terminal punctuation.


In [ ]:
FILLER_WORDS = [
    r"\bum+\b", r"\buh+\b", r"\ber+\b", r"\byou know\b",
    r"\blike\b(?=,)", r"\bi mean\b",
]

def clean_transcription(text):
    """Clean a raw Whisper transcription before passing it to the reasoning model."""
    cleaned = text.strip()

    # Remove filler words (case-insensitive)
    for pattern in FILLER_WORDS:
        cleaned = re.sub(pattern, "", cleaned, flags=re.IGNORECASE)

    # Collapse multiple spaces into one
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    # Fix stray spaces before punctuation
    cleaned = re.sub(r"\s+([,.!?])", r"\1", cleaned)

    # Ensure the sentence ends with terminal punctuation
    if cleaned and cleaned[-1] not in ".!?":
        cleaned += "."

    return cleaned


# Demonstrate cleaning on the smoke-test transcription
print("Before cleaning:", test_result["text"])
print("After cleaning: ", clean_transcription(test_result["text"]))


## Step 9: Free GPU Memory Before Loading the Reasoning Model

Free-tier Colab GPUs (T4, 16 GB) have limited memory. Keeping both the Whisper model and a large
language model resident on the GPU at the same time is unnecessary for this pipeline and risks an
out-of-memory (OOM) crash. Since all transcription work is finished, we explicitly delete the
Whisper model and clear the CUDA cache before loading the reasoning LLM.


In [ ]:
del whisper_model
gc.collect()
torch.cuda.empty_cache()

if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated(0) / (1024 ** 3)
    print(f"GPU memory allocated after cleanup: {allocated:.3f} GB")
print("Whisper model unloaded. GPU memory freed for the reasoning model.")


## Step 10: Reasoning Model Selection

We load **`unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit`** using Unsloth's dynamic 4-bit
quantization.

**Why this model is suitable:**

- **Reasoning-focused:** DeepSeek-R1-Distill models are distilled from DeepSeek-R1, a model
  specifically trained to produce structured, step-by-step chain-of-thought reasoning rather than
  short, unexplained answers — a good fit for a pipeline whose stated goal is "generate logical
  answers."
- **Fits free-tier GPU memory:** At 4-bit precision the 8B-parameter model occupies roughly 5-6 GB
  of VRAM, which comfortably fits on a 16 GB T4 alongside the activation memory needed for
  generation, especially after the Whisper model has been unloaded in the previous step.
- **Unsloth compatibility:** Unsloth provides a pre-quantized, ready-to-load `bnb-4bit` version of
  this checkpoint together with fast custom CUDA kernels for attention and MLP layers, giving
  noticeably faster inference than a plain `transformers` + `bitsandbytes` load.
- **Llama-compatible tokenizer/architecture:** Because the model is distilled onto a Llama
  architecture, it benefits from the same mature tooling (chat templates, KV-cache, sampling
  utilities) used throughout the Llama ecosystem.

If you would prefer a smaller/faster model (for example, on a slower connection or a smaller GPU),
`unsloth/Llama-3.2-3B-Instruct-unsloth-bnb-4bit` or
`unsloth/Qwen2.5-7B-Instruct-unsloth-bnb-4bit` are both drop-in alternatives — simply change
`MODEL_NAME` in the next cell. Note that only DeepSeek-R1-Distill style models produce the explicit
`<think>...</think>` reasoning trace shown later in this notebook; general instruct models will
still answer correctly but without that visible reasoning trace.


In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # generous context window; plenty for short transcriptions + reasoning output

print(f"Loading reasoning model: {MODEL_NAME} ...")
reasoning_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect best dtype for the current GPU
    load_in_4bit=True,   # dynamic 4-bit quantization
)

# Switch the model into fast inference mode (enables Unsloth's optimized generation path)
FastLanguageModel.for_inference(reasoning_model)

print("Reasoning model loaded and ready for inference.")


## Step 11: Reasoning Inference Function

We build the prompt using the tokenizer's own `apply_chat_template` method (the standard,
well-supported Hugging Face approach) rather than any custom chat-template import, and generate a
response with sampling parameters tuned for coherent, focused reasoning.


In [ ]:
def generate_reasoning_response(question, max_new_tokens=512, temperature=0.6, top_p=0.9):
    """
    Send a cleaned transcription/question to the reasoning model and return its response.

    Args:
        question (str): The cleaned transcribed text to reason about.
        max_new_tokens (int): Maximum tokens to generate.
        temperature (float): Sampling temperature.
        top_p (float): Nucleus sampling parameter.

    Returns:
        dict with keys: "response", "generation_seconds"
    """
    messages = [
        {
            "role": "user",
            "content": (
                "Answer the following question with clear, step-by-step reasoning, "
                "then give a final concise answer.\n\n"
                f"Question: {question}"
            ),
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(reasoning_model.device)

    start_time = time.time()
    with torch.no_grad():
        output_ids = reasoning_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    elapsed = time.time() - start_time

    generated_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    response_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return {
        "response": response_text,
        "generation_seconds": round(elapsed, 2),
    }


# Quick smoke test
smoke_test = generate_reasoning_response("What is two plus two?")
print("Generation time:", smoke_test["generation_seconds"], "seconds")
print("Response:")
print(smoke_test["response"])


## Step 12: Full Speech-to-Reasoning Pipeline

This function ties every previous step together into a single call:

`audio file -> Whisper transcription -> cleaning -> reasoning model -> final answer`

Note that at this point the Whisper model has already been unloaded (Step 9), so this function
re-loads a lightweight Whisper instance only if needed. For this notebook's test run we instead
keep the transcription step separate below to avoid reloading Whisper repeatedly; in a production
script you would typically wrap both models behind a small class that loads/unloads them as
needed, exactly as demonstrated in Steps 6-9.


In [ ]:
def run_speech_to_reasoning_pipeline(audio_path, whisper_model_instance, verbose=False):
    """
    Full pipeline: transcribe an audio file and generate a reasoning response.

    Args:
        audio_path (str): Path to the input audio file.
        whisper_model_instance: A loaded Whisper model to use for transcription.
        verbose (bool): Whether to print intermediate steps.

    Returns:
        dict with keys: audio_path, raw_text, cleaned_text, response,
                        transcription_seconds, generation_seconds
    """
    transcription = transcribe_audio(audio_path, model=whisper_model_instance)
    cleaned_text = clean_transcription(transcription["text"])
    reasoning = generate_reasoning_response(cleaned_text)

    result = {
        "audio_path": audio_path,
        "raw_text": transcription["text"],
        "cleaned_text": cleaned_text,
        "response": reasoning["response"],
        "transcription_seconds": transcription["duration_seconds"],
        "generation_seconds": reasoning["generation_seconds"],
    }

    if verbose:
        print("Audio file:", audio_path)
        print("Raw transcription:", result["raw_text"])
        print("Cleaned transcription:", result["cleaned_text"])
        print("Reasoning response:")
        print(result["response"])

    return result

print("Pipeline function defined: run_speech_to_reasoning_pipeline()")


## Step 13: Test the Pipeline on Multiple Sample Audio Files

We reload a fresh Whisper `base` model (since it was unloaded in Step 9) and run the complete
pipeline over every sample audio clip generated earlier, displaying:

- The original audio (playable widget)
- The transcribed text
- The final reasoning response from the LLM


In [ ]:
print("Reloading Whisper model for the test run ...")
test_whisper_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)

all_results = []

for idx, audio_path in enumerate(audio_paths, start=1):
    display(Markdown(f"### Sample {idx}: `{os.path.basename(audio_path)}`"))
    display(Audio(audio_path))

    result = run_speech_to_reasoning_pipeline(audio_path, test_whisper_model)
    all_results.append(result)

    print("Transcribed text  :", result["cleaned_text"])
    print(f"(transcription took {result['transcription_seconds']}s, "
          f"reasoning took {result['generation_seconds']}s)")
    print()
    print("Reasoning response:")
    print(textwrap.fill(result["response"], width=100))
    print("=" * 100)


## Step 14: Save Results

Finally, we save all transcriptions and reasoning responses to a text file inside the `outputs`
folder created in Step 4, so the results can be attached to the submission report.


In [ ]:
output_path = os.path.join(OUTPUT_DIR, "pipeline_results.txt")

with open(output_path, "w") as f:
    for idx, result in enumerate(all_results, start=1):
        f.write(f"Sample {idx}: {os.path.basename(result['audio_path'])}\n")
        f.write(f"Transcribed text: {result['cleaned_text']}\n")
        f.write(f"Reasoning response: {result['response']}\n")
        f.write(f"Transcription time (s): {result['transcription_seconds']}\n")
        f.write(f"Generation time (s): {result['generation_seconds']}\n")
        f.write("-" * 80 + "\n")

print(f"Results saved to: {output_path}")


## Summary

This notebook demonstrated a complete, self-contained speech-to-reasoning pipeline:

1. **Speech recognition** with OpenAI Whisper, handling audio of any length via its internal
   windowed transcription and timestamp stitching.
2. **Transcription cleaning** to remove filler words and normalize punctuation before the text is
   used as a model prompt.
3. **GPU memory management**: the Whisper model is explicitly unloaded before the reasoning model
   is loaded, avoiding out-of-memory errors on the free-tier T4 GPU.
4. **Quantized reasoning inference** using Unsloth's dynamic 4-bit quantization of
   DeepSeek-R1-Distill-Llama-8B, giving step-by-step reasoning answers with a small memory
   footprint and fast generation.
5. **End-to-end testing** across multiple sample audio clips, with original audio, transcription,
   and final reasoning response all displayed together and saved to disk.

See the accompanying `README.md`, `IMPLEMENTATION_GUIDE.md`, and `TROUBLESHOOTING.md` files for
setup instructions, folder structure, and common error resolutions.
